In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import calendar

# =========================
# 0. 既存データ
# =========================
lineup_df = pd.read_csv("npb_2024_lineup_all.csv")

def parse_lineup_date(x):
    s = str(x).strip()
    s = s.replace("月", "/").replace("日", "")
    s = re.sub(r"\(.*?\)", "", s)      # (土) など削除
    s = re.sub(r"[^0-9/]", "", s)      # 数字と / 以外を削除

    m = re.search(r"(\d{1,2})/(\d{1,2})", s)
    if m:
        month = int(m.group(1))
        day = int(m.group(2))
        return pd.Timestamp(f"2025-{month:02d}-{day:02d}")
    return pd.NaT

lineup_df["日付"] = lineup_df["日付"].apply(parse_lineup_date)
lineup_df["4番"] = lineup_df["4番"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

main_fourth_df = pd.DataFrame({
    "球団": ["DeNA", "オリックス", "ソフトバンク", "ヤクルト", "ロッテ", "中日",
           "巨人", "広島", "日本ハム", "楽天", "西武", "阪神"],
    "選手名": ["牧 秀悟", "杉本 裕太郎", "山川 穂高", "オスナ", "山本 大斗", "細川 成也",
            "岡本 和真", "末包 昇大", "野村 佑希", "ボイト", "ネビン", "佐藤 輝明"]
})
main_fourth_df["選手名"] = main_fourth_df["選手名"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

TEAM_CODE_MAP = {
    "巨人": "G", "阪神": "T", "DeNA": "DB", "広島": "C", "ヤクルト": "S", "中日": "D",
    "ソフトバンク": "H", "日本ハム": "F", "オリックス": "B", "ロッテ": "M", "楽天": "E", "西武": "L",
}
REVERSE_TEAM_CODE_MAP = {v: k for k, v in TEAM_CODE_MAP.items()}

HEADERS = {"User-Agent": "Mozilla/5.0"}

CALENDAR_PAGES = {
    "04": "https://npb.jp/bis/eng/2024/calendar/index_04.html",
    "05": "https://npb.jp/bis/eng/2024/calendar/index_05.html",
    "06": "https://npb.jp/bis/eng/2024/calendar/index_06.html",
    "07": "https://npb.jp/bis/eng/2024/calendar/index_07.html",
    "08": "https://npb.jp/bis/eng/2024/calendar/index_08.html",
    "09": "https://npb.jp/bis/eng/2024/calendar/index_09.html",
    "10": "https://npb.jp/bis/eng/2024/calendar/index_10.html",
}

def parse_score_line(line: str):
    line = re.sub(r"\s+", " ", line.strip())
    m = re.fullmatch(r"([A-Z]+) (\d+) - (\d+) ([A-Z]+)", line)
    if not m:
        return None
    return {
        "team1": m.group(1),
        "score1": int(m.group(2)),
        "score2": int(m.group(3)),
        "team2": m.group(4),
    }

def safe_timestamp(year, month, day):
    last_day = calendar.monthrange(year, month)[1]
    if 1 <= day <= last_day:
        return pd.Timestamp(year=year, month=month, day=day)
    return None

def scrape_month_results(page_code: str, url: str) -> pd.DataFrame:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = r.apparent_encoding

    soup = BeautifulSoup(r.text, "html.parser")
    lines = [x.strip() for x in soup.get_text("\n").splitlines() if x.strip()]

    rows = []
    current_date = None

    for line in lines:
        if re.fullmatch(r"\d{1,2}", line):
            day = int(line)

            if page_code == "04":
                if 28 <= day <= 31:
                    dt = safe_timestamp(2025, 3, day)
                else:
                    dt = safe_timestamp(2025, 4, day)
            else:
                dt = safe_timestamp(2025, int(page_code), day)

            if dt is not None:
                current_date = dt
            continue

        parsed = parse_score_line(line)
        if parsed and current_date is not None:
            rows.append({
                "日付": current_date,
                "team1": parsed["team1"],
                "score1": parsed["score1"],
                "team2": parsed["team2"],
                "score2": parsed["score2"],
                "raw": line,
            })

    return pd.DataFrame(rows)

# =========================
# 1. カレンダー取得
# =========================
all_months = []

for page_code, url in CALENDAR_PAGES.items():
    try:
        print(f"取得中: {page_code}")
        mdf = scrape_month_results(page_code, url)
        print(f"  {len(mdf)}試合分")
        if not mdf.empty:
            all_months.append(mdf)
        time.sleep(1)
    except Exception as e:
        print(f"[ERROR] {page_code}: {e}")

games_df = pd.concat(all_months, ignore_index=True)

team_game_rows = []
for _, row in games_df.iterrows():
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team1"]),
        "得点": row["score1"],
        "raw": row["raw"],
    })
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team2"]),
        "得点": row["score2"],
        "raw": row["raw"],
    })

team_scores_df = pd.DataFrame(team_game_rows).dropna(subset=["球団"])

# 確認
print("\nlineup_df 日付欠損数:", lineup_df["日付"].isna().sum())
print(lineup_df[["球団", "日付", "4番"]].head(10))

# =========================
# 2. 結合
# =========================
merged_list = []

for _, r in main_fourth_df.iterrows():
    team = r["球団"]
    player = r["選手名"]

    subset = lineup_df[
        (lineup_df["球団"] == team) &
        (lineup_df["4番"] == player)
    ][["球団", "日付", "4番"]].copy()

    subset = subset.rename(columns={"4番": "選手名"})

    subset = subset.merge(
        team_scores_df[["球団", "日付", "得点", "raw"]],
        on=["球団", "日付"],
        how="left"
    )

    merged_list.append(subset)

player_game_scores_df = pd.concat(merged_list, ignore_index=True)

print("\n結合後 得点欠損数:", player_game_scores_df["得点"].isna().sum())
print(player_game_scores_df.head(20))

# =========================
# 3. 集計
# =========================
summary_df = (
    player_game_scores_df
    .groupby(["球団", "選手名"], as_index=False)
    .agg({"得点": ["count", "sum", "mean"]})
)

summary_df.columns = ["球団", "選手名", "4番試合数", "合計得点", "平均得点"]
summary_df["平均得点"] = summary_df["平均得点"].round(3)

print("\n=== 代表4番打者が4番で出場した試合の平均得点 ===")
print(summary_df.sort_values("平均得点", ascending=False))

player_game_scores_df.to_csv(
    "npb_2024_main_fourth_batter_game_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "npb_2024_main_fourth_batter_avg_runs_when_batting_4th.csv",
    index=False,
    encoding="utf-8-sig"
)
print("\n保存完了")

取得中: 04
  157試合分
取得中: 05
  140試合分
取得中: 06
  134試合分
取得中: 07
  126試合分
取得中: 08
  150試合分
取得中: 09
  134試合分
取得中: 10
  38試合分

lineup_df 日付欠損数: 0
   球団         日付     4番
0  阪神 2025-03-29  大山 悠輔
1  阪神 2025-03-30  大山 悠輔
2  阪神 2025-03-31  大山 悠輔
3  阪神 2025-04-02  大山 悠輔
4  阪神 2025-04-03  大山 悠輔
5  阪神 2025-04-04  大山 悠輔
6  阪神 2025-04-05  大山 悠輔
7  阪神 2025-04-06  大山 悠輔
8  阪神 2025-04-07  大山 悠輔
9  阪神 2025-04-09  大山 悠輔

結合後 得点欠損数: 9
      球団         日付   選手名   得点          raw
0   DeNA 2025-03-29  牧 秀悟  4.0   DB 4 - 3 C
1   DeNA 2025-03-29  牧 秀悟  1.0  D 11 - 1 DB
2   DeNA 2025-03-30  牧 秀悟  6.0   DB 6 - 1 C
3   DeNA 2025-03-30  牧 秀悟  2.0   D 1 - 2 DB
4   DeNA 2025-03-31  牧 秀悟  1.0   DB 1 - 5 C
5   DeNA 2025-04-02  牧 秀悟  5.0   T 3 - 5 DB
6   DeNA 2025-04-03  牧 秀悟  2.0   T 5 - 2 DB
7   DeNA 2025-04-04  牧 秀悟  3.0   T 2 - 3 DB
8   DeNA 2025-04-05  牧 秀悟  2.0   G 1 - 2 DB
9   DeNA 2025-04-06  牧 秀悟  6.0   G 4 - 6 DB
10  DeNA 2025-04-07  牧 秀悟  0.0   G 3 - 0 DB
11  DeNA 2025-04-09  牧 秀悟  1.0   DB 1 - 3 D
12  DeNA 202